<a href="https://colab.research.google.com/github/atanyeri/Problem-Set-1-/blob/main/Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [63]:
# Clean consumer transactions

# lognormal parameters describe log(values): median = exp(mean)
MU, SIGMA = np.log(40), 0.7   # median basket ~ $40, right skewed
N_PER_YEAR = 12000  # 24,000 clean rows in total
N_CUSTOMERS = 8000

clean = pd.DataFrame({
    "customer_id": rng.integers(1, N_CUSTOMERS + 1, size=2 * N_PER_YEAR),
    "year": np.repeat([1, 2], N_PER_YEAR),
    "basket_value": rng.lognormal(MU, SIGMA, size=2 * N_PER_YEAR)
})
clean.head()

,customer_id,year,basket_value
0,715,1,48.615013
1,6192,1,17.129922
2,5237,1,30.125701
3,3512,1,25.590299
4,3465,1,20.059344


In [64]:
# The truth: the consumer average per year, recorded before contamination
true_avg = clean.groupby("year")["basket_value"].mean()
true_avg

,basket_value
year,
1,51.849914
2,51.636122


In [65]:
# Contamination 1: B2B orders

# few in number, large in value, and the business grows in year 2
B2B_COUNTS = {1: 150, 2: 400}
b2b_parts = []
for yr, n in B2B_COUNTS.items():
    b2b_parts.append(pd.DataFrame({
        "customer_id": rng.integers(N_CUSTOMERS + 1, N_CUSTOMERS + 201, size=n),
        "year": yr,
        "basket_value": rng.lognormal(np.log(1500), 0.6, size=n),
        "status": "completed"
    }))
b2b = pd.concat(b2b_parts)
b2b.head()

,customer_id,year,basket_value,status
0,8032,1,4333.341112,completed
1,8108,1,934.623048,completed
2,8100,1,501.802368,completed
3,8060,1,1258.213068,completed
4,8180,1,6789.303178,completed


In [66]:
# Contamination 2: what gets logged changes between year 1 and year 2

# year 1: only completed orders are logged
# year 2: cancelled orders start appearing in the table too, skewed larger
N_CANCELLED = 1800
cancelled = pd.DataFrame({
    "customer_id": rng.integers(1, N_CUSTOMERS + 1, size=N_CANCELLED),
    "year": 2,
    "basket_value": rng.lognormal(np.log(70), 0.8, size=N_CANCELLED),
    "status": "cancelled"
})
clean["status"] = "completed"
logged = pd.concat([clean, b2b, cancelled])
logged = logged.sample(frac=1, random_state=42).reset_index(drop=True)
len(logged)


26350

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [67]:
# The dashboard's number vs the truth
dashboard = logged.groupby("year")["basket_value"].mean()

summary = pd.DataFrame({
    "true_avg": true_avg,
    "dashboard_avg": dashboard
})

summary["gap_$"] = summary["dashboard_avg"] - summary["true_avg"]
summary["gap_%"] = summary["gap_$"] / summary["true_avg"]


true_growth = true_avg[2] / true_avg[1] - 1
dash_growth = dashboard[2] / dashboard[1] - 1
print(f"True growth yoy: {true_growth:.2%}")
print(f"Dashboard growth yoy: {dash_growth:.2%}")
print(f"Phantom Growth: {dash_growth - true_growth:+.2%} points")


True growth yoy: -0.41%
Dashboard growth yoy: 46.37%
Phantom Growth: +46.78% points


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [68]:
# How much of the gap is the skew and the B2B tail?
is_b2b = logged["customer_id"] > N_CUSTOMERS
is_cancelled = logged["status"] == "cancelled"

def yearly_mean(mask):
  return logged[mask].groupby("year")["basket_value"].mean()

naive = dashboard.copy()
no_b2b = yearly_mean(~is_b2b)   # b2b removed, cancelled still in
no_canc = yearly_mean(~is_cancelled)  # cancelled removed, b2b still in
neither = yearly_mean(~is_b2b & ~is_cancelled)  # both removed

gap = naive - true_avg

# skew: how far the consumer mean sits above median in both years
clean_by_year = clean.groupby("year")["basket_value"]
print("Consumer mean vs median (skew):")
print(pd.DataFrame({'mean': clean_by_year.mean(), 'median': clean_by_year.median()}).round(2))
print("Unexplained gap after removing B2B and cancelled:", (neither - true_avg).abs().max())

# B2B tail: effects interact, so average over both removal orders
b2b_contrib = ((naive - no_b2b) + (no_canc - neither)) / 2
print("\n B2B contribution to the gap:")
print(pd.DataFrame({'gap_$':gap, 'b2b_$':b2b_contrib, 'b2b_share':b2b_contrib / gap}))



Consumer mean vs median (skew):
       mean  median
year               
1     51.85   40.37
2     51.64   40.18
Unexplained gap after removing B2B and cancelled: 0.0

 B2B contribution to the gap:
          gap_$      b2b_$  b2b_share
year                                 
1     23.541171  23.541171   1.000000
2     58.713258  56.848754   0.968244


In [69]:
# How much is the change in what was logged?

# cancelled orders, same two-order average
canc_contrib = ((naive - no_canc) + (no_b2b - neither)) / 2
print("\n Cancelled contribution to the gap:")
print(pd.DataFrame({'gap_$':gap, 'canc_$':canc_contrib, 'canc_share':canc_contrib / gap}))

# the two pieces should add up to whole gap
print("\n B2B + cancelled - gap:", (b2b_contrib + canc_contrib - gap).abs().max())

# same split for the phantom growth
def growth(s): return s[2] / s[1] - 1

phantom = growth(naive) - growth(true_avg)
b2b_g = ((growth(naive) - growth(no_b2b)) + (growth(no_canc) - growth(neither))) / 2
canc_g = ((growth(naive) - growth(no_canc)) + (growth(no_b2b) - growth(neither))) / 2

print(f"\n Phantom growth: {phantom:+.2%} points")
print(f"from B2B tail: {b2b_g:+.2%} points")
print(f"from cancelled: {canc_g:+.2%} points")


 Cancelled contribution to the gap:
          gap_$    canc_$  canc_share
year                                 
1     23.541171  0.000000    0.000000
2     58.713258  1.864504    0.031756

 B2B + cancelled - gap: 0.0

 Phantom growth: +46.78% points
from B2B tail: +42.56% points
from cancelled: +4.22% points


Mechanism, Contribution to the gap, How you isolated it

Skew and B2B tail
$56.85 of the $58.71 year-2 gap (96.8%), and $23.54 of $23.54 in year 1. In growth terms, 42.56 of the 46.78 points of phantom growth (91%). The skew alone contributes $0.
Removed B2B orders (customer_id > N_CUSTOMERS) and recomputed the yearly mean. Because the two mechanisms interact, I averaged the effect over both removal orders. For the skew, I compared the consumer mean to the median in each year (51.85 vs 40.37, 51.64 vs 40.18). The $11.5 distance is the same in both years, so it creates no gap.


Change in what was logged
$1.86 of the year-2 gap (3.2%), and $0 in year 1. In growth terms, 4.22 of the 46.78 points (9%).
Removed cancelled orders (status == "cancelled") and recomputed the mean, again averaged over both removal orders. The effect ranges from -$2.07 (removed first) to +$5.80 (removed after B2B), so $1.86 is the central figure.

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [70]:
# Median, trimmed mean, and the mean after your B2B exclusion rule

# explicit B2B rule found without the flags: 4 robust SDs above the median on the log scale
log_v = np.log(logged["basket_value"])
med, mad = log_v.median(), stats.median_abs_deviation(log_v, scale="normal")
threshold = np.exp(med + 4 * mad)
rule = logged["basket_value"] > threshold
print(f"threshold ${threshold:,.0f}; caught {(rule & is_b2b).sum()} of {is_b2b.sum()} B2B, "
      f"false positives {(rule & ~is_b2b).sum()}")

by_year = lambda d: d.groupby("year")["basket_value"]
results = pd.DataFrame({
    "true": true_avg,
    "naive_mean": naive,
    "median": by_year(logged).median(),
    "trimmed_10%": by_year(logged).apply(lambda s: stats.trim_mean(s, 0.1)),
    "excl_rule_mean": by_year(logged[~rule]).mean(),
    "excl_rule+completed": by_year(logged[~rule & ~is_cancelled]).mean()
})
print(results)
print(results.apply(growth))

# what a trimmed mean gives on the clean data just to see its built-in bias
print("true 10% trimmed mean:", by_year(clean).apply(lambda s: stats.trim_mean(s, 0.1)).tolist())

threshold $813; caught 482 of 550 B2B, false positives 2
           true  naive_mean     median  trimmed_10%  excl_rule_mean  \
year                                                                  
1     51.849914   75.391085  40.794583    46.059978       52.815585   
2     51.636122  110.349380  43.632252    50.357747       58.973309   

      excl_rule+completed  
year                       
1               52.815585  
2               53.682959  
true                  -0.004123
naive_mean             0.463693
median                 0.069560
trimmed_10%            0.093308
excl_rule_mean         0.116589
excl_rule+completed    0.016423
dtype: float64
true 10% trimmed mean: [45.21159522819038, 44.66467393734007]


Estimator, Value, The assumption that makes it right, When it would be wrong

Median
$40.79 -> $43.63 (+7.0%)
Client wants the typical shopper's spend, and the contamination is a minority of rows far out in one tail, so it cant move in the middle
When the client needs dollars per order, since the median ignores order size. It also fails when the contamination is a large share of rows, as the 2200 extra high-value rows in year 2 shift the ranks so it shows +7.0% when the true median is flatter (40.37->40.18)

Trimmed mean:
$46.06 -> $50.36 (+9.3%)
Contamination is under 10% of rows and a symmetric trim is acceptable. B2B is 1.2% of rows in year 1 and 2.8% in year 2, so it's covered
When the data is skewed, because it also cuts good cunsumer orders from the top. On clean data it gives $45.21 and $44.66, about $7 below the true mean, so the level doesn't match the historical metric. It also can't remove cancelled orders, so growth is still +9.3% against a true trimmed growth of -1.2%


Mean after B2B exclusion
$52.82 -> $58.97 (+11.77%) with the value rule alone, $52.82 -> $54.68 (+1.6%) with completed-only orders added |B2B orders can be told apart from consumer orders by a rule that is stable across years, ideally an account type
When the rule is value-based and the two populations overlap. The $813 rule leaks 68 B2B orders, with 22 in year 1 and 46 in year 2. The leak grows with volume. It's also wrong if B2B belongs in the metric.

**Which number should the dashboard show?** Commit to one and defend it.

The dashboard should show the mean basket value of completed consumer orders, with B2B excluded by account type. It's the only option that still measures average consumer basket in dollars, and together the B2B exclusion and completed-only filter cut phantom growth from 46.4% to about 1.6%, which would go to zero with an account flag instead of a size cutoff. The median or trimmed mean would change the quantity being reported and can't tell a cancelled order from a completed one.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [71]:
import os
os.makedirs("src", exist_ok=True)

In [72]:
%%writefile src/basket_metrics.py
"""YOUR ONE-LINE DESCRIPTION OF THE MODULE."""

import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: missingness, dtype, skew and outlier count."""
    raise NotImplementedError


def robust_mean(x, method: str = "median") -> float:
    """A robust centre for x. method: "median", "trimmed" or your exclusion rule."""
    raise NotImplementedError


if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    pass

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [73]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance

<module 'basket_metrics' from '/content/src/basket_metrics.py'>

---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [74]:
# The analysis that settles it

**What you found, including if it changes your Phase 2 recommendation:**

*Your answer here.*

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

*Your slide here.*

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors